# 01探索性数据分析

该note用于检查恒星类型预测项目的原始数据，包括数据规模、字段结构、目标变量、缺失值、重复值、变量类型和初步分布情况。

In [6]:
#导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [7]:
# 定义数据路径
DATA_DIR = Path("../data/raw")

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
submission_path = DATA_DIR / "sample_submission.csv"

print(train_path)
print(test_path)
print(submission_path)

..\data\raw\train.csv
..\data\raw\test.csv
..\data\raw\sample_submission.csv


In [8]:
# 加载数据
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

train shape: (577347, 12)
test shape: (247435, 11)
sample_submission shape: (247435, 2)


In [ ]:
# 显示训练数据的前几行
train.head()
# 显示测试数据的前几行
test.head()
# 显示提交文件的前几行
sample_submission.head()

In [ ]:
# 显示列名
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSample submission columns:")
print(sample_submission.columns.tolist())

列名	                    含义
id	              每条样本的唯一编号，只用于对应结果，通常不作为模型特征
alpha	          赤经，天体在天空中的横向坐标，类似经度
delta	          赤纬，天体在天空中的纵向坐标，类似纬度
u, g, r, i, z	  五个光学波段/滤镜下的亮度或星等特征，可反映颜色和光谱差异
redshift	      红移，表示光谱波长被拉长的程度，常和天体远近、退行速度有关
spectral_type	  光谱类型，分类特征
galaxy_population 星系/天体族群类别，分类特征
class	          目标标签，也就是模型要预测的类别

In [6]:
# 找出训练数据中存在但测试数据中不存在的列，这些列可能是目标变量
possible_target_cols = [col for col in train.columns if col not in test.columns]

print("Possible target columns:")
print(possible_target_cols)

Possible target columns:
['class']


In [ ]:
# 显示训练数据的基本信息
train.info()
# 显示测试数据的基本信息
test.info()

In [ ]:
# 显示训练数据的统计信息
train.describe()

In [ ]:
# 检查缺失值
missing_train = train.isnull().sum().sort_values(ascending=False)
missing_test = test.isnull().sum().sort_values(ascending=False)

print("Missing values in train:")
print(missing_train[missing_train > 0])

print("\nMissing values in test:")
print(missing_test[missing_test > 0])

In [10]:
# 检查重复值
print("Duplicated rows in train:", train.duplicated().sum())
print("Duplicated rows in test:", test.duplicated().sum())

Duplicated rows in train: 0
Duplicated rows in test: 0


In [11]:
# 手动设置目标列
target_col = possible_target_cols[0]
print("Target column:", target_col)
# 显示目标列的分布
train[target_col].value_counts()
# 看比例
train[target_col].value_counts(normalize=True)

Target column: class


class
GALAXY    0.653818
QSO       0.202899
STAR      0.143283
Name: proportion, dtype: float64

也就是说训练集不是均衡分布，GALAXY 占了接近三分之二，STAR 只有约 14%。模型如果只追求 accuracy，可能会偏向预测成 GALAXY。

In [ ]:
#画目标变量分布图
train[target_col].value_counts().sort_index().plot(kind="bar")

plt.title("Target Distribution")
plt.xlabel(target_col)
plt.ylabel("Count")
plt.show()

In [ ]:
#区分数值变量和类别变量
id_cols = ["id"] if "id" in train.columns else []

feature_cols = [col for col in train.columns if col not in id_cols + [target_col]]

num_cols = train[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = train[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

print("ID columns:", id_cols)
print("Feature columns:", feature_cols)
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

数值变量
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
alpha、delta：天体坐标
u, g, r, i, z：不同波段的亮度/星等
redshift：红移

分类变量
['spectral_type', 'galaxy_population']
spectral_type	  光谱类型，分类特征
galaxy_population 星系/天体族群类别，分类特征


In [ ]:
#数值变量分布图
from pathlib import Path

fig_dir = Path("../reports/figures/numeric_distributions")
fig_dir.mkdir(parents=True, exist_ok=True)

for col in num_cols:
    train[col].hist(bins=30)

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.savefig(fig_dir / f"distribution_{col}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# 可视化类别变量的分布
for col in cat_cols:
    print(f"\n{col}")
    print(train[col].value_counts())

In [ ]:
# 特征和目标变量的关系，并保存为 Excel 表格
TABLE_DIR = Path("../reports/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

excel_path = TABLE_DIR / "numeric_features_by_target_describe.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for col in num_cols:
        desc = train.groupby(target_col)[col].describe()
        display(desc)
        desc.to_excel(writer, sheet_name=col[:31])

print(f"Saved: {excel_path}")